# Sistema de Triaje Predictivo en Urgencias Hospitalarias
## Análisis Exploratorio de Datos (EDA), Ingeniería de Características y Benchmark de Modelos

Este notebook documenta el pipeline completo de ciencia de datos para clasificar la prioridad de triaje (, , ) a partir de síntomas y constantes vitales de pacientes.

**Estructura del Notebook:**
1. Carga y exploración inicial del dataset.
2. Pruebas bioestadísticas de hipótesis (Chi-cuadrado, V de Cramér, Kruskal-Wallis).
3. Ingeniería de características clínicas (Índice de Shock, Presión Arterial Media, binarización de síntomas).
4. Validación cruzada de 6 algoritmos de clasificación.
5. Evaluación final en conjunto de prueba independiente y análisis de importancia de variables.
6. Exportación del pipeline reproducible.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuración de estilo visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]

# Ruta a datos
data_path = "../data/disease_diagnosis.csv" if os.path.exists("../data/disease_diagnosis.csv") else "data/disease_diagnosis.csv"
df = pd.read_csv(data_path)
print(f"Dimensiones: {df.shape[0]} pacientes, {df.shape[1]} variables.")
df.head(5)

### 1. Exploración Inicial y Distribución de la Variable Objetivo ()
Observamos la proporción de pacientes por nivel de urgencia.

In [ ]:
print(df["Severity"].value_counts())
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="Severity", order=["Mild", "Moderate", "Severe"], palette=["#2ecc71", "#f39c12", "#e74c3c"], ax=ax)
ax.set_title("Distribución de Niveles de Severidad en Admisión", fontsize=12, fontweight="bold")
ax.set_xlabel("Nivel de Severidad")
ax.set_ylabel("Número de Pacientes")
for p in ax.patches:
    ax.annotate(f"{p.get_height()} ({p.get_height()/len(df)*100:.1f}%)", (p.get_x() + p.get_width()/2., p.get_height()/2),
                ha="center", va="center", color="white", fontweight="bold")
plt.show()

### 2. Análisis Estadístico de Síntomas vs Severidad
Binarizamos los 8 síntomas canónicos y calculamos la prueba de Chi-cuadrado y la V de Cramér para medir el grado de asociación con la gravedad.

In [ ]:
sintomas = sorted(list(set(df["Symptom_1"].dropna().unique().tolist() + df["Symptom_2"].dropna().unique().tolist() + df["Symptom_3"].dropna().unique().tolist())))

symptom_stats = []
for sym in sintomas:
    has_sym = ((df["Symptom_1"] == sym) | (df["Symptom_2"] == sym) | (df["Symptom_3"] == sym)).astype(int)
    ct = pd.crosstab(has_sym, df["Severity"])
    chi2, p_val, _, _ = stats.chi2_contingency(ct)
    n = ct.to_numpy().sum()
    cv = np.sqrt((chi2/n) / (min(ct.shape)-1))
    prev_mild = (has_sym[df["Severity"]=="Mild"].mean() * 100)
    prev_mod = (has_sym[df["Severity"]=="Moderate"].mean() * 100)
    prev_sev = (has_sym[df["Severity"]=="Severe"].mean() * 100)
    symptom_stats.append({
        "Síntoma": sym, "Chi2": round(chi2, 2), "p-valor": f"{p_val:.2e}",
        "V de Cramér": round(cv, 4), "% Leve": round(prev_mild, 1),
        "% Moderado": round(prev_mod, 1), "% Severo": round(prev_sev, 1)
    })

df_sintomas = pd.DataFrame(symptom_stats).sort_values(by="V de Cramér", ascending=False)
df_sintomas

### 3. Signos Vitales: Temperatura y Saturación de Oxígeno por Severidad
Graficamos las distribuciones para constatar las diferencias fisiológicas observadas en las pruebas de Kruskal-Wallis.

### 3.1 Auditoría de Outliers: Explicación Matemática y Detección de Anomalías Multivariadas
**Pregunta clave:** ¿Existen outliers en el dataset?
- **Univariadamente (Tukey IQR):** No hay outliers (0.0%). Las variables siguen distribuciones uniformes truncadas con curtosis ~ -1.20, donde matemáticamente ningún valor puede superar  + 1.5 	imes IQR$.
- **Multivariadamente e Inconsistencias Fisiológicas:** Se detectan **156 casos** donde la presión diastólica es mayor o igual a la sistólica ( \ge PAS$), producto de la generación sintética independiente.

In [ ]:
# 1. Comprobación univariada de curtosis y límites de Tukey
num_cols = ["Age", "Heart_Rate_bpm", "Body_Temperature_C", "BP_Systolic", "BP_Diastolic", "Oxygen_Saturation_%"]
bp_split = df["Blood_Pressure_mmHg"].str.split("/", expand=True).astype(int)
df["BP_Systolic"] = bp_split[0]
df["BP_Diastolic"] = bp_split[1]
df["Pulse_Pressure"] = df["BP_Systolic"] - df["BP_Diastolic"]
df["Shock_Index"] = df["Heart_Rate_bpm"] / df["BP_Systolic"]

print("--- CURTOSIS Y OUTLIERS IQR ---")
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    n_outliers = len(df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)])
    print(f"{col:<22}: Curtosis={df[col].kurtosis():>6.2f} | Outliers IQR={n_outliers}")

# 2. Anomalías Fisiológicas Reales (PAS <= PAD)
inconsistentes = df[df["Pulse_Pressure"] <= 0]
print(f"
Inconsistencias Fisiológicas (PAD >= PAS): {len(inconsistentes)} casos ({len(inconsistentes)/len(df)*100:.2f}%)")
print(f"Presión de Pulso Mínima: {df["Pulse_Pressure"].min()} mmHg")

# 3. Gráfico de dispersión de inconsistencias
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df, x="BP_Systolic", y="BP_Diastolic", hue=(df["Pulse_Pressure"] <= 0).map({True: "PAD >= PAS (Inconsistente)", False: "Normal (PAS > PAD)"}), palette=["#2980b9", "#e74c3c"], alpha=0.7, ax=ax)
ax.plot([80, 180], [80, 180], "r--", label="PAS = PAD")
ax.set_title("Detección de Anomalías: 156 Casos con PAD >= PAS")
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="Severity", y="Body_Temperature_C", order=["Mild", "Moderate", "Severe"], palette=["#2ecc71", "#f39c12", "#e74c3c"], ax=axes[0])
axes[0].set_title("Temperatura Corporal (°C) por Nivel de Severidad")
axes[0].set_ylabel("Temperatura (°C)")
axes[0].axhline(38.0, color="red", linestyle="--", alpha=0.7, label="Umbral Fiebre (38.0°C)")
axes[0].legend()

sns.boxplot(data=df, x="Severity", y="Oxygen_Saturation_%", order=["Mild", "Moderate", "Severe"], palette=["#2ecc71", "#f39c12", "#e74c3c"], ax=axes[1])
axes[1].set_title("Saturación de Oxígeno (%) por Nivel de Severidad")
axes[1].set_ylabel("SatO2 (%)")
axes[1].axhline(95.0, color="red", linestyle="--", alpha=0.7, label="Umbral Hipoxemia (95%)")
axes[1].legend()
plt.tight_layout()
plt.show()

### 4. Ingeniería de Características Clínicas
Construimos el transformador que extrae la presión arterial desglosada (, ), el  (Índice de Shock),  (Presión Arterial Media) y las banderas clínicas de riesgo.

In [ ]:
# Importamos el módulo de preprocesamiento del proyecto
sys.path.insert(0, os.path.abspath(".."))
from src.data_preprocessing import ClinicalFeatureExtractor, preparar_datos_entrenamiento

X, y, features = preparar_datos_entrenamiento(data_path)
print(f"Conjunto de características generado: {X.shape[1]} variables.")
X[["shock_index", "map_pressure", "flag_hypoxia", "flag_fever", "flag_tachycardia"]].head(5)

### 5. Benchmark de Algoritmos (Validación Cruzada 5-Fold)
Evaluamos 6 clasificadores representativos del curso (Regresión Logística, k-NN, SVM, Random Forest, XGBoost, MLP).

In [ ]:
from src.train_and_evaluate import benchmark_modelos
df_benchmark = benchmark_modelos(X, y)
df_benchmark

### 6. Evaluación en Conjunto de Prueba y Matriz de Confusión
Cargamos el pipeline final entrenado y verificamos el desempeño sobre el conjunto de test independiente.

In [ ]:
import joblib
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

model_path = "../models/triaje_model.joblib" if os.path.exists("../models/triaje_model.joblib") else "models/triaje_model.joblib"
pipeline = joblib.load(model_path)

X_raw = df.drop(columns=["Patient_ID", "Diagnosis", "Treatment_Plan", "Severity"])
y_raw = df["Severity"]
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

y_pred = pipeline.predict(X_test_raw)
print(classification_report(y_test, y_pred))

labels = ["Mild", "Moderate", "Severe"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Matriz de Confusión - Conjunto de Prueba (N=400)")
plt.xlabel("Predicción del Modelo")
plt.ylabel("Severidad Real del Paciente")
plt.show()

### 7. Importancia de Características del Modelo Final
Inspeccionamos la contribución porcentual de cada variable en el bosque aleatorio.

In [ ]:
rf_clf = pipeline.named_steps["classifier"]
extractor = pipeline.named_steps["feature_extractor"]
feat_names = X.columns.tolist()
importances = pd.Series(rf_clf.feature_importances_, index=feat_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.head(10).values * 100, y=importances.head(10).index, palette="viridis")
plt.title("Top 10 Variables más Importantes (Random Forest)", fontsize=12, fontweight="bold")
plt.xlabel("Importancia Relativa (%)")
plt.ylabel("Variable Clínica / Síntoma")
plt.tight_layout()
plt.show()